In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print('TensorFlow version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.20.0
GPU: []


In [ ]:
import kagglehub

path = kagglehub.dataset_download('pankaj4321/fer-2013-facial-expression-dataset')
print('Dataset path:', path)

base_dir  = path
train_dir = os.path.join(base_dir, 'train')
val_dir   = os.path.join(base_dir, 'val')
test_dir  = os.path.join(base_dir, 'test')

categories = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

def count_images(directory):
    counts = {}
    for cat in categories:
        p = os.path.join(directory, cat)
        counts[cat] = len(os.listdir(p)) if os.path.exists(p) else 0
    return counts

print('Train:', count_images(train_dir))
print('Val  :', count_images(val_dir))
print('Test :', count_images(test_dir))

Dataset path: /root/.cache/kagglehub/datasets/pankaj4321/fer-2013-facial-expression-dataset/versions/1
Train: {'Angry': 3995, 'Disgust': 436, 'Fear': 4097, 'Happy': 7215, 'Sad': 4830, 'Surprise': 3171, 'Neutral': 4965}
Val  : {'Angry': 467, 'Disgust': 56, 'Fear': 496, 'Happy': 895, 'Sad': 653, 'Surprise': 415, 'Neutral': 607}
Test : {'Angry': 491, 'Disgust': 55, 'Fear': 528, 'Happy': 879, 'Sad': 594, 'Surprise': 416, 'Neutral': 626}


In [ ]:
IMG_SIZE   = (96, 96)   # FER-2013 originals are 48×48; 96×96 gives MobileNetV2 more detail

BATCH_SIZE = 64

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('Class indices:', train_gen.class_indices)
print(f'Train batches: {len(train_gen)}  |  Val batches: {len(val_gen)}  |  Test batches: {len(test_gen)}')

Found 28709 images belonging to 7 classes.
Found 3589 images belonging to 7 classes.
Found 3589 images belonging to 7 classes.
Class indices: {'Angry': 0, 'Disgust': 1, 'Fear': 2, 'Happy': 3, 'Neutral': 4, 'Sad': 5, 'Surprise': 6}
Train batches: 449  |  Val batches: 57  |  Test batches: 57


In [ ]:
labels = train_gen.classes

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weight_dict = dict(enumerate(class_weights_array))

print('Class weights:')
idx_to_class = {v: k for k, v in train_gen.class_indices.items()}
for idx, w in class_weight_dict.items():
    print(f'  {idx_to_class[idx]}: {w:.4f}')

Class weights:
  Angry: 1.0266
  Disgust: 9.4066
  Fear: 1.0010
  Happy: 0.5684
  Neutral: 0.8260
  Sad: 0.8491
  Surprise: 1.2934


In [ ]:
base_model = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

output = Dense(7, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

#new weight = old weight − (learning rate × gradient)

model.compile(optimizer=Adam(learning_rate=1e-3),loss='categorical_crossentropy',metrics=['accuracy'])


trainable     = sum(tf.size(w).numpy() for w in model.trainable_weights)
non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
print(f'Trainable params:     {trainable:,}')
print(f'Non-trainable params: {non_trainable:,}')

Trainable params:     362,503
Non-trainable params: 2,258,752


In [ ]:
CHECKPOINT_PATH = 'trained_model.keras'

callbacks = [
    ModelCheckpoint(filepath=CHECKPOINT_PATH,monitor='val_accuracy',save_best_only=True,verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=10,restore_best_weights=True,verbose=1),
    ReduceLROnPlateau(monitor='val_loss',factor=0.5, patience=5, min_lr=1e-7, verbose=1)
]


In [ ]:
PHASE1_EPOCHS = 30

hist1 = model.fit(train_gen, steps_per_epoch=len(train_gen), epochs=PHASE1_EPOCHS, validation_data=val_gen, validation_steps=len(val_gen), class_weight=class_weight_dict,callbacks=callbacks)

Epoch 1/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 552ms/step - accuracy: 0.2553 - loss: 2.1961
Epoch 1: val_accuracy improved from None to 0.38228, saving model to trained_model.keras

Epoch 1: finished saving model to trained_model.keras
449/449 ━━━━━━━━━━━━━━━━━━━━ 280s 608ms/step - accuracy: 0.2837 - loss: 1.9799 - val_accuracy: 0.3823 - val_loss: 1.6257 - learning_rate: 0.0010
Epoch 2/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 557ms/step - accuracy: 0.3400 - loss: 1.6986
Epoch 2: val_accuracy improved from 0.38228 to 0.41711, saving model to trained_model.keras

Epoch 2: finished saving model to trained_model.keras
449/449 ━━━━━━━━━━━━━━━━━━━━ 321s 607ms/step - accuracy: 0.3483 - loss: 1.6654 - val_accuracy: 0.4171 - val_loss: 1.5420 - learning_rate: 0.0010
Epoch 3/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 550ms/step - accuracy: 0.3773 - loss: 1.5932
Epoch 3: val_accuracy did not improve from 0.41711
449/449 ━━━━━━━━━━━━━━━━━━━━ 320s 602ms/step - accuracy: 0.3752 - loss: 1.6063 - val_accuracy: 0.3918 - 

In [ ]:
model.load_weights('trained_model.keras')

In [ ]:

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_p2 = [
    ModelCheckpoint('finetuned_model.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=10,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=5, min_lr=1e-8, verbose=1)
]

hist2 = model.fit(
    train_gen,
    steps_per_epoch=len(train_gen),
    epochs=20,
    validation_data=val_gen,
    validation_steps=len(val_gen),
    class_weight=class_weight_dict,
    callbacks=callbacks_p2
)

Epoch 1/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 794ms/step - accuracy: 0.3290 - loss: 1.7839
Epoch 1: val_accuracy improved from None to 0.42435, saving model to finetuned_model.keras

Epoch 1: finished saving model to finetuned_model.keras
449/449 ━━━━━━━━━━━━━━━━━━━━ 396s 851ms/step - accuracy: 0.3403 - loss: 1.7482 - val_accuracy: 0.4244 - val_loss: 1.5426 - learning_rate: 1.0000e-05
Epoch 2/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 792ms/step - accuracy: 0.3618 - loss: 1.6347
Epoch 2: val_accuracy did not improve from 0.42435
449/449 ━━━━━━━━━━━━━━━━━━━━ 378s 842ms/step - accuracy: 0.3663 - loss: 1.6275 - val_accuracy: 0.4188 - val_loss: 1.5234 - learning_rate: 1.0000e-05
Epoch 3/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 796ms/step - accuracy: 0.3717 - loss: 1.5734
Epoch 3: val_accuracy did not improve from 0.42435
449/449 ━━━━━━━━━━━━━━━━━━━━ 381s 849ms/step - accuracy: 0.3759 - loss: 1.5629 - val_accuracy: 0.4096 - val_loss: 1.5292 - learning_rate: 1.0000e-05
Epoch 4/20
449/449 ━━━━━━━━━━━━━━━━━━━━

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix

model = load_model('finetuned_model.keras')
#model = load_model('/content/sample_data/finetuned_model.keras')

test_loss, test_acc = model.evaluate(test_gen)
print(f"Test Accuracy: {test_acc:.4f}")

test_gen.reset()
preds = model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes
labels = list(test_gen.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=labels))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('emotion_confusion_matrix.png')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(hist1.history['accuracy'], label='Phase 1 Train')
axes[0].plot(hist1.history['val_accuracy'], label='Phase 1 Val')
axes[0].plot(range(30, 30 + len(hist2.history['accuracy'])), hist2.history['accuracy'], label='Phase 2 Train')
axes[0].plot(range(30, 30 + len(hist2.history['val_accuracy'])), hist2.history['val_accuracy'], label='Phase 2 Val')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(hist1.history['loss'], label='Phase 1 Train')
axes[1].plot(hist1.history['val_loss'], label='Phase 1 Val')
axes[1].plot(range(30, 30 + len(hist2.history['loss'])), hist2.history['loss'], label='Phase 2 Train')
axes[1].plot(range(30, 30 + len(hist2.history['val_loss'])), hist2.history['val_loss'], label='Phase 2 Val')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_curves.png')
plt.show()
